In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/dataset/online_retail_II.csv",

    header=True,
    inferSchema=True
)

In [0]:
df.printSchema()

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)



In [0]:
df.show(10)

+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
| 489434|    85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00| 6.95|    13085.0|United Kingdom|
| 489434|   79323P|  PINK CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|   79323W| WHITE CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|    22041|"RECORD FRAME 7""...|      48|2009-12-01 07:45:00|  2.1|    13085.0|United Kingdom|
| 489434|    21232|STRAWBERRY CERAMI...|      24|2009-12-01 07:45:00| 1.25|    13085.0|United Kingdom|
| 489434|    22064|PINK DOUGHNUT TRI...|      24|2009-12-01 07:45:00| 1.65|    13085.0|United Kingdom|
| 489434|    21871| SAVE THE PLANET MUG|      24|2009-12-01 07:45:00| 1.2

In [0]:
df.count()

1067371

In [0]:
df.columns

['Invoice',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'Price',
 'Customer ID',
 'Country']

In [0]:
df.describe().display()

summary,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country
count,1067371,1067371,1062989,1067371,1067371,824364,1067371
mean,537608.1499316233,28350.201592689715,21848.25,9.9388984711033,4.649387727416074,15324.63850435002,null
stddev,26662.45044690487,17968.479697262945,922.9197780233488,172.7057940767533,123.55305872146253,1697.4644503793093,null
min,489434,10002,DOORMAT UNION JACK GUNS AND ROSES,-80995,-53594.36,12346.0,Australia
max,C581569,m,wrongly sold sets,80995,38970.0,18287.0,West Indies


Checking for Missing Values 

In [0]:
from pyspark.sql.functions import col, count, when

df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show(10)

+-------+---------+-----------+--------+-----------+-----+-----------+-------+
|Invoice|StockCode|Description|Quantity|InvoiceDate|Price|Customer ID|Country|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+
|      0|        0|       4382|       0|          0|    0|     243007|      0|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+



In [0]:
print("Rows :", df.count())

print("Distinct :", df.distinct().count())

Rows : 1067371
Distinct : 1033036


##Data Cleaning

In [0]:
print("Before:", df.count())

df = df.dropDuplicates()

print("After:", df.count())

Before: 1067371
After: 1033036


Removing missing cutomer IDs

In [0]:

df = df.filter(col("Customer ID").isNotNull())

In [0]:
df.selectExpr("count(*) as Rows").show()

+------+
|  Rows|
+------+
|797885|
+------+



Removing Cancelled invoices

In [0]:
df.filter(col("Invoice").startswith("C")).show(10, False)

+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|Description                        |Quantity|InvoiceDate        |Price|Customer ID|Country       |
+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+--------------+
|C489570|15056N   |EDWARDIAN PARASOL NATURAL          |-1      |2009-12-01 13:28:00|5.95 |15967.0    |United Kingdom|
|C490120|22333    |RETRO SPORT PARTY BAG + STICKER SET|-16     |2009-12-03 17:52:00|1.65 |14277.0    |France        |
|C490120|22138    |BAKING SET 9 PIECE RETROSPOT       |-9      |2009-12-03 17:52:00|4.95 |14277.0    |France        |
|C490798|21121    |SET/10 RED SPOTTY PARTY CANDLES    |-24     |2009-12-08 11:51:00|1.25 |14277.0    |France        |
|C490866|85065    |CREAM SWEETHEART TRAYS             |-1      |2009-12-08 12:51:00|12.75|17213.0    |United Kingdom|
|C490939|22352    |LUNCHBOX WITH CUTLERY RETROSPOT    |-

In [0]:
df = df.filter(~col("Invoice").startswith("C"))

Removing invalid quantity and prices

In [0]:
df = df.filter(col("Quantity") > 0) 
df = df.filter(col("Price") > 0)

In [0]:
print("Final Rows:", df.count())

df.describe().display()

Final Rows: 779425


summary,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country
count,779425,779425,779425,779425,779425,779425,779425
mean,537426.828586458,28218.015114655915,null,13.489369727683869,3.2184879853755954,15320.360460595952,null
stddev,26901.62962737051,17724.991431787563,null,145.85581409978133,29.676139695711342,1695.6927751567978,null
min,489434,10002,DOORMAT UNION JACK GUNS AND ROSES,1,0.001,12346.0,Australia
max,581587,TEST002,ZINC WIRE SWEETHEART LETTER TRAY,80995,10953.5,18287.0,West Indies


##Feature Engineering

Revenue

In [0]:
df = df.withColumn(
    "Revenue",
    col("Quantity") * col("Price")
)

In [0]:
df.select(
    "Quantity",
    "Price",
    "Revenue"
).show(5)

+--------+-----+------------------+
|Quantity|Price|           Revenue|
+--------+-----+------------------+
|       2| 6.75|              13.5|
|      32| 2.55|              81.6|
|       1|  2.1|               2.1|
|       3| 5.95|             17.85|
|       3| 4.95|14.850000000000001|
+--------+-----+------------------+
only showing top 5 rows


Year

In [0]:
from pyspark.sql.functions import year
df = df.withColumn(
    "Year",
    year("InvoiceDate")
)
df.select("InvoiceDate","Year").show(5)

+-------------------+----+
|        InvoiceDate|Year|
+-------------------+----+
|2009-12-01 09:46:00|2009|
|2009-12-01 10:06:00|2009|
|2009-12-01 11:21:00|2009|
|2009-12-01 11:37:00|2009|
|2009-12-01 11:41:00|2009|
+-------------------+----+
only showing top 5 rows


Month

In [0]:
from pyspark.sql.functions import month
df = df.withColumn(
    "Month",
    month("InvoiceDate")

)

In [0]:
from pyspark.sql.functions import date_format
df = df.withColumn(
    "Month_Name",
    date_format("InvoiceDate","MMMM")
)

Quater

In [0]:
from pyspark.sql.functions import quarter

df = df.withColumn(
    "Quarter",
    quarter("InvoiceDate")
)

Week Days

In [0]:
df = df.withColumn(
    "Weekday",
    date_format("InvoiceDate","EEEE")
)

Hour

In [0]:
from pyspark.sql.functions import hour

df = df.withColumn(
    "Hour",
    hour("InvoiceDate")
)

Weekend

In [0]:


df = df.withColumn(
    "Weekend",
    when(
        col("Weekday").isin("Saturday","Sunday"),
        "Weekend"
    ).otherwise("Weekday")
)

Invoice month

In [0]:
from pyspark.sql.functions import trunc

df = df.withColumn(
    "InvoiceMonth",
    trunc("InvoiceDate","Month")
)

df = df.withColumn(
    "InvoiceMonth",
    trunc("InvoiceDate","Month")
)

In [0]:
df.printSchema()

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Month_Name: string (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Weekday: string (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- Weekend: string (nullable = false)
 |-- InvoiceMonth: date (nullable = true)



##Customer Segmentaion Using WIndow Function

Latest date

In [0]:
from pyspark.sql.functions import max

latest_date = df.select(max("InvoiceDate")).collect()[0][0]

print(latest_date)

2011-12-09 12:50:00


Customer Level RFM

In [0]:
from pyspark.sql.functions import (
    max,
    countDistinct,
    sum,
    datediff,
    lit,
    round
)

rfm = (
    df.groupBy("Customer ID")
      .agg(
          max("InvoiceDate").alias("LastPurchase"),
          countDistinct("Invoice").alias("Frequency"),
          round(sum("Revenue"),2).alias("Monetary")
      )
)

Recency

In [0]:
rfm = rfm.withColumn(
    "Recency",
    datediff(
        lit(latest_date),
        rfm["LastPurchase"]
    )
)

In [0]:
rfm.show(10)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-------------------+---------+---------+-------+-------+-------+-------+---------+-------------------+
|Customer ID|       LastPurchase|Frequency| Monetary|Recency|R_Score|F_Score|M_Score|RFM_Score|            Segment|
+-----------+-------------------+---------+---------+-------+-------+-------+-------+---------+-------------------+
|    18102.0|2011-12-09 11:50:00|      145|580987.04|      0|      5|      1|      1|      511|Potential Loyalists|
|    14646.0|2011-12-08 12:12:00|      151|528602.52|      1|      5|      1|      1|      511|Potential Loyalists|
|    14156.0|2011-11-30 10:54:00|      156|313437.62|      9|      5|      1|      1|      511|Potential Loyalists|
|    14911.0|2011-12-08 15:54:00|      398|291420.81|      1|      5|      1|      1|      511|Potential Loyalists|
|    17450.0|2011-12-01 13:29:00|       51|244784.25|      8|      5|      1|      1|      511|Potential Loyalists|
|    13694.0|2011-12-06 09:32:00|      143|195640.69|      3|      5|   

Window Function 

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import ntile

In [0]:
recency_window = Window.orderBy("Recency")

frequency_window = Window.orderBy(col("Frequency").desc())

monetary_window = Window.orderBy(col("Monetary").desc())

Assign Score

In [0]:
rfm = rfm.withColumn(
    "R_Score",
    6 - ntile(5).over(recency_window)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
rfm = rfm.withColumn(
    "F_Score",
    ntile(5).over(frequency_window)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
rfm = rfm.withColumn(
    "M_Score",
    ntile(5).over(monetary_window)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
rfm.select(
    "Customer ID",
    "Recency",
    "Frequency",
    "Monetary",
    "R_Score",
    "F_Score",
    "M_Score"
).show(20, False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-------+---------+---------+-------+-------+-------+
|Customer ID|Recency|Frequency|Monetary |R_Score|F_Score|M_Score|
+-----------+-------+---------+---------+-------+-------+-------+
|18102.0    |0      |145      |580987.04|5      |1      |1      |
|14646.0    |1      |151      |528602.52|5      |1      |1      |
|14156.0    |9      |156      |313437.62|5      |1      |1      |
|14911.0    |1      |398      |291420.81|5      |1      |1      |
|17450.0    |8      |51       |244784.25|5      |1      |1      |
|13694.0    |3      |143      |195640.69|5      |1      |1      |
|17511.0    |2      |60       |172132.87|5      |1      |1      |
|16446.0    |0      |2        |168472.5 |5      |3      |1      |
|16684.0    |4      |55       |147142.77|5      |1      |1      |
|12415.0    |24     |28       |144458.37|4      |1      |1      |
|15061.0    |3      |127      |126389.02|5      |1      |1      |
|16029.0    |38     |107      |117763.62|4      |1      |1      |
|17949.0  

In [0]:
from pyspark.sql.functions import concat

rfm = rfm.withColumn(
    "RFM_Score",
    concat(
        col("R_Score"),
        col("F_Score"),
        col("M_Score")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Customer Segmentaion

In [0]:
from pyspark.sql.functions import when

rfm = rfm.withColumn(
    "Segment",
    when(
        (col("R_Score") >= 4) &
        (col("F_Score") >= 4) &
        (col("M_Score") >= 4),
        "Champions"
    )
    .when(
        (col("R_Score") >= 3) &
        (col("F_Score") >= 3),
        "Loyal Customers"
    )
    .when(
        col("R_Score") >= 4,
        "Potential Loyalists"
    )
    .when(
        col("R_Score") <= 2,
        "At Risk"
    )
    .otherwise("Others")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
rfm.groupBy("Segment") \
   .count() \
   .orderBy(col("count").desc()) \
   .show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------+-----+
|            Segment|count|
+-------------------+-----+
|            At Risk| 2350|
|Potential Loyalists| 1591|
|    Loyal Customers| 1222|
|             Others|  479|
|          Champions|  236|
+-------------------+-----+



Top Customer

In [0]:
rfm.orderBy(col("Monetary").desc()) \
   .select(
       "Customer ID",
       "Monetary",
       "Frequency",
       "Recency",
       "Segment"
   ) \
   .show(20, False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+---------+---------+-------+-------------------+
|Customer ID|Monetary |Frequency|Recency|Segment            |
+-----------+---------+---------+-------+-------------------+
|18102.0    |580987.04|145      |0      |Potential Loyalists|
|14646.0    |528602.52|151      |1      |Potential Loyalists|
|14156.0    |313437.62|156      |9      |Potential Loyalists|
|14911.0    |291420.81|398      |1      |Potential Loyalists|
|17450.0    |244784.25|51       |8      |Potential Loyalists|
|13694.0    |195640.69|143      |3      |Potential Loyalists|
|17511.0    |172132.87|60       |2      |Potential Loyalists|
|16446.0    |168472.5 |2        |0      |Loyal Customers    |
|16684.0    |147142.77|55       |4      |Potential Loyalists|
|12415.0    |144458.37|28       |24     |Potential Loyalists|
|15061.0    |126389.02|127      |3      |Potential Loyalists|
|16029.0    |117763.62|107      |38     |Potential Loyalists|
|17949.0    |117314.08|118      |1      |Potential Loyalists|
|15311.0

Parquet format

In [0]:
df.write \
    .mode("overwrite") \
    .partitionBy("Year", "Month") \
    .parquet("/Volumes/workspace/default/dataset/")

In [0]:
parquet_df = spark.read.parquet(
    "/Volumes/workspace/default/dataset/"
)

parquet_df.show(5)

+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+------------------+----------+-------+---------+----+-------+------------+----+-----+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|           Revenue|Month_Name|Quarter|  Weekday|Hour|Weekend|InvoiceMonth|Year|Month|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+------------------+----------+-------+---------+----+-------+------------+----+-----+
| 579885|    82486|3 DRAWER ANTIQUE ...|       2|2011-11-30 17:37:00| 8.95|    15444.0|United Kingdom|              17.9|  November|      4|Wednesday|  17|Weekday|  2011-11-01|2011|   11|
| 579872|    23275|SET OF 3 HANGING ...|      12|2011-11-30 16:54:00| 1.25|    14085.0|United Kingdom|              15.0|  November|      4|Wednesday|  16|Weekday|  2011-11-01|2011|   11|
| 579865|    23012|GLASS APOTHECARY ...|       3|2011-11-30 

In [0]:
parquet_df.printSchema()

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- Month_Name: string (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Weekday: string (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- Weekend: string (nullable = true)
 |-- InvoiceMonth: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)



In [0]:
jan_data = parquet_df.filter(
    (col("Year") == 2011) &
    (col("Month") == 1)
)

print(jan_data.count())

20988
